In [1]:
from playwright.async_api import async_playwright
import asyncio
import re
from dataclasses import dataclass
from pydantic import BaseModel, Field
from typing import Optional, List
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os


In [3]:
brave_params = {
    "command": "npx", 
    "args": ["-y", "@modelcontextprotocol/server-brave-search"], 
    "env": {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
}

playwright_params = {
    "command": "npx",
    "args": ["@playwright/mcp@latest"]
}


In [4]:
class SearchResults(BaseModel):
    """URLs found from Brave Search"""
    product_urls: List[str] = Field(description="List of BestBuy product URLs")

class Deal(BaseModel):
    """
    A class to Represent a Deal with a summary description
    """
    product_description: str = Field(
        description="Full product description: Brand + Name + Features"
    )
    price: float = Field(
        description="The current sale price in USD"
    )
    url: str = Field(
        description="The BestBuy product URL"
    )

class DealSelection(BaseModel):
    """
    A class to Represent a list of Deals
    """
    deals: List[Deal] = Field(
        description="List of deals with detailed description and clear price"
    )

In [5]:
def search_instructions():
    return """
You are a web search agent. Your job is to find product URLs on BestBuy.

STEPS:
1. Use brave_web_search with: keyword + "site:bestbuy.com/product"
2. Extract product URLs from results
3. Return ONLY URLs matching: https://www.bestbuy.com/product/...

IMPORTANT:
- Do NOT return search page URLs
- Return up to 15 product URLs
"""

In [7]:
# Cell 5: Test Brave Search MCP
keyword = "Smart tv"

with trace("Step1_BraveSearch"):
    async with MCPServerStdio(params=brave_params, client_session_timeout_seconds=60) as brave_server:
        
        search_agent = Agent(
            name="SearchAgent",
            instructions=search_instructions(),
            model="gpt-5-nano",
            mcp_servers=[brave_server],
            output_type=SearchResults
        )
        
        result = await Runner.run(
            search_agent,
            f"Search for {keyword} on BestBuy",
            max_turns=30
        )
        
        urls = result.final_output.product_urls

print(f"Found {len(urls)} URLs:")

Found 14 URLs:


In [8]:
# Cell: Filter Sale URLs (Simple - No LLM)
import requests
from bs4 import BeautifulSoup
import time
from typing import List

def is_on_sale(url: str, timeout: int = 10) -> bool:
    """
    Check if BestBuy product is on sale.
    
    Args:
        url: BestBuy product URL
        timeout: Request timeout in seconds
        
    Returns:
        True if product is on sale, False otherwise
    """
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=timeout)
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Check for sale indicators
        savings_elem = soup.find(attrs={"data-testid": "price-block-total-savings-text"})
        comp_value_elem = soup.find(attrs={"data-lu-target": "comp_value"})
        
        return savings_elem is not None or comp_value_elem is not None
        
    except Exception as e:
        print(f"Error checking {url}: {e}")
        return False


def filter_sale_urls(urls: List[str]) -> List[str]:
    """
    Filter URLs to keep only products on sale.
    
    Args:
        urls: List of BestBuy product URLs
        
    Returns:
        List of URLs for products on sale
    """
    sale_urls = []
    
    for i, url in enumerate(urls, 1):
        if is_on_sale(url):
            sale_urls.append(url)
            print(f"[{i}/{len(urls)}] SALE")
        else:
            print(f"[{i}/{len(urls)}] Skip")
        time.sleep(0.05)
    
    return sale_urls

In [ ]:
print(f"Input: {len(urls)} URLs")

sale_urls = filter_sale_urls(urls)

print(f"Output: {len(sale_urls)} sale URLs")

for url in sale_urls:
    print(url)

Input: 14 URLs
[1/14] SALE
[2/14] SALE
[3/14] SALE
[4/14] SALE
[5/14] SALE
[6/14] SALE
[7/14] SALE
[8/14] SALE
[9/14] SALE
[10/14] SALE
[11/14] SALE
[12/14] SALE
[13/14] SALE
[14/14] SALE
Output: 14 sale URLs
https://www.bestbuy.com/product/westinghouse-24-class-smart-tv-hd-xumo-tv-with-voice-remote-flat-screen-led-television/J3LL89C69K
https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW
https://www.bestbuy.com/product/tcl-65-class-qm8k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQZ3T
https://www.bestbuy.com/product/samsung-65-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5VV
https://www.bestbuy.com/product/tcl-65-class-qm5k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQWZ4
https://www.bestbuy.com/product/tcl-50-class-qm5k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTT2Y8
https://www.bestbuy.com/product/tcl-55-qm6k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYT

In [10]:
sale_urls

['https://www.bestbuy.com/product/westinghouse-24-class-smart-tv-hd-xumo-tv-with-voice-remote-flat-screen-led-television/J3LL89C69K',
 'https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW',
 'https://www.bestbuy.com/product/tcl-65-class-qm8k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQZ3T',
 'https://www.bestbuy.com/product/samsung-65-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5VV',
 'https://www.bestbuy.com/product/tcl-65-class-qm5k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQWZ4',
 'https://www.bestbuy.com/product/tcl-50-class-qm5k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTT2Y8',
 'https://www.bestbuy.com/product/tcl-55-qm6k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQW5C',
 'https://www.bestbuy.com/product/samsung-70-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2VF78',
 'https://www.bestbuy.com/product/tcl-55-class-qm5k-series-4k-uhd-hdr-qd

In [11]:
def create_product_description(name: str, brand: Optional[str], features: str) -> str:
    """
    Combine: Brand + Name + Features → product_description
    """
    parts = []
    
    if brand:
        parts.append(f"Brand: {brand}")
    
    parts.append(name)
    
    if features and len(features) > 10:
        features_clean = features[:1500].strip()
        parts.append(f"Features: {features_clean}")
    
    return "\n".join(parts)

In [12]:
# Cell: Scrape BestBuy products → DealSelection
async def scrape_bestbuy_to_deals(urls: List[str], headless: bool = False) -> DealSelection:
    """
    Scrape BestBuy products and return as DealSelection.
    
    Args:
        urls: List of sale product URLs
        headless: Run browser in headless mode
        
    Returns:
        DealSelection containing List[Deal]
    """
    deals = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=headless,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox']
        )
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080}
        )
        page = await context.new_page()
        
        for i, url in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] Scraping...")
            
            try:
                await page.goto(url, timeout=60000, wait_until="domcontentloaded")
                await page.wait_for_timeout(2000)
                
                # Extract Name
                name_elem = page.locator("h1.h4")
                name = await name_elem.text_content() if await name_elem.count() > 0 else "Unknown"
                name = name.strip() if name else "Unknown"
                
                # Extract Brand
                brand_elem = page.locator('div[data-component-name="ProductHeader"] a.c-button-link')
                brand = await brand_elem.first.text_content() if await brand_elem.count() > 0 else None
                brand = brand.strip() if brand else None
                
                # Extract Sale Price
                price_elem = page.locator('[data-testid="price-block-customer-price"] span')
                price_text = await price_elem.first.text_content() if await price_elem.count() > 0 else "$0"
                price_match = re.search(r'[\d,]+\.?\d*', price_text.replace(',', ''))
                price = float(price_match.group()) if price_match else 0.0
                
                # Click Features and extract
                features = ""
                features_btn = page.locator('button:has(h3:text("Features"))')
                
                if await features_btn.count() > 0:
                    await features_btn.first.click()
                    try:
                        await page.locator('[data-testid="brix-sheet-content"]').wait_for(timeout=5000)
                        features_elem = page.locator('[data-testid="brix-sheet-content"]')
                        features = await features_elem.first.text_content() or ""
                    except:
                        pass
                    await page.keyboard.press("Escape")
                    await page.wait_for_timeout(500)
                
                # Create Deal
                product_description = create_product_description(name, brand, features)
                deal = Deal(
                    product_description=product_description,
                    price=price,
                    url=url
                )
                deals.append(deal)
                print(f"{name[:40]}... | ${price}")
                
            except Exception as e:
                print(f" Error: {e}")
                continue
        
        await browser.close()
    
    return DealSelection(deals=deals)

In [13]:
deal_selection = await scrape_bestbuy_to_deals(sale_urls, headless=False)

print("\n" + "="*60)
print(f"✅ DealSelection with {len(deal_selection.deals)} deals:")
print("="*60)

for i, deal in enumerate(deal_selection.deals, 1):
    print(f"\n--- Deal {i} ---")
    print(f"💰 Price: ${deal.price}")
    print(f"🔗 URL: {deal.url}")

[1/14] Scraping...
Westinghouse - 24” Class Smart TV, HD Xu... | $79.99
[2/14] Scraping...
Samsung - 55" Class U7900 Series UHD 4K ... | $279.99
[3/14] Scraping...
TCL - 65" Class QM8K Series 4K UHD QD-Mi... | $999.99
[4/14] Scraping...
Samsung - 65" Class U7900 Series UHD 4K ... | $329.99
[5/14] Scraping...
TCL - 65" Class QM5K Series 4K UHD QD-Mi... | $0.0
[6/14] Scraping...
TCL - 50" Class QM5K Series 4K UHD HDR Q... | $299.99
[7/14] Scraping...
TCL - 55" QM6K Series 4K UHD HDR QD Mini... | $449.99
[8/14] Scraping...
Samsung - 70" Class U7900 Series UHD 4K ... | $399.99
[9/14] Scraping...
TCL - 55" Class QM5K Series 4K UHD HDR Q... | $329.99
[10/14] Scraping...
TCL - 40" Class Q3K Series 1080P FHD QLE... | $149.99
[11/14] Scraping...
TCL - 65" Class QM6K Series 4K UHD HDR Q... | $529.99
[12/14] Scraping...
TCL - 85" Class QM5K Series 4K UHD QD-Mi... | $899.99
[13/14] Scraping...
TCL - 85" Class QM6K Series 4K UHD HDR Q... | $999.99
[14/14] Scraping...
TCL - 98" Class QM8K Series 4K 